# Sanity check — reconstructed intra-US MRIOT

Empirical verification that the bilateral IO links rebuilt in
[v3_construction.ipynb](v3_construction.ipynb) are *economically plausible*. We do
not re-derive the construction; we load a finished build
(`grav_fric_v3.1_RAS/IOT_<year>.npz`) and look at whether characteristic state
economies show up where they should.

**What we check**
- **Part A** — automatically rank `state × sector` cells by *specialization*
  (location quotient) to surface the characteristic economies (TX oil & gas,
  MI motor vehicles, NV accommodation, IA farms, …).
- **Part B** — for each characteristic state, draw its full structure:
  intra-state sector² heatmap, interstate trade links (exports / imports by
  partner state and good), production-cost decomposition, final-demand
  decomposition, and the tax decomposition.
- **Part C** — numeric accounting checks (row = good supply, column = industry
  output, trade balance) so the pictures rest on verified identities.

**Schema reminder** (grav_fric). Index space is `(region, item)` flattened as
`r*S + k`, `n=51` regions × `S=71` sectors. `Z` is `(n·S, n·S)` with
**rows = (origin region, *good* supplied)** and **columns = (destination region,
*industry* buying)**. `F` is `(n·S, n·3)`, columns = (region, [C, I, G]). `VA`,
`EX`, `M`, `M_interm`, `taxes`, `tax_prod`, `tariff`, `tls_int`, `tls_fd`, `TLS`
are flat `(n·S,)`.


In [ ]:
# ── Setup: load a finished build + sector labels ─────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from paths import ROOT
VERSION = 'v3.1_RAS'        # full-resolution final table (TLS + VA convention)
YEAR    = '2017'

npz = np.load(ROOT / f'data/interim/IOT/IOT_USA/grav_fric_{VERSION}/IOT_{YEAR}.npz',
              allow_pickle=True)
regions = [str(r) for r in npz['regions']]
sectors = [str(s) for s in npz['sectors']]
n, S = len(regions), len(sectors)
ridx = {r: i for i, r in enumerate(regions)}
sidx = {s: i for i, s in enumerate(sectors)}

# flat blocks
Z, F   = npz['Z'], npz['F']
VA, EX = npz['VA'], npz['EX']
M, Mi  = npz['M'], npz['M_interm']
taxes  = npz['taxes']
tax_prod, tariff = npz['tax_prod'], npz['tariff']
tls_int, tls_fd  = npz['tls_int'], npz['tls_fd']

# reshaped views: Z4[origin r, good g, dest r2, industry s] ; F4[r, g, r2, C/I/G]
Z4 = Z.reshape(n, S, n, S)
F4 = F.reshape(n, S, n, 3)

# (region, item) helpers
def col(r):                       # column slice (industry side) for region r
    return slice(ridx[r] * S, ridx[r] * S + S)
def vec(a, r):                    # a flat (n*S,) array -> (S,) for region r
    return a.reshape(n, S)[ridx[r]]

# human-readable sector names (windc_desc), short labels for dense axes
conc = pd.read_csv(ROOT / 'data/raw/correspondence/windc_oecd_concordance.csv')
DESC = conc.drop_duplicates('windc_sector').set_index('windc_sector')['windc_desc'].to_dict()
def name(code):                   # 'oil' -> 'oil — Oil and gas extraction (211)'
    return f"{code} — {DESC.get(code, code)}"

STATE = {  # USPS -> full name, for figure titles
 'AL':'Alabama','AK':'Alaska','AZ':'Arizona','AR':'Arkansas','CA':'California',
 'CO':'Colorado','CT':'Connecticut','DE':'Delaware','DC':'D.C.','FL':'Florida',
 'GA':'Georgia','HI':'Hawaii','ID':'Idaho','IL':'Illinois','IN':'Indiana',
 'IA':'Iowa','KS':'Kansas','KY':'Kentucky','LA':'Louisiana','ME':'Maine',
 'MD':'Maryland','MA':'Massachusetts','MI':'Michigan','MN':'Minnesota',
 'MS':'Mississippi','MO':'Missouri','MT':'Montana','NE':'Nebraska','NV':'Nevada',
 'NH':'New Hampshire','NJ':'New Jersey','NM':'New Mexico','NY':'New York',
 'NC':'North Carolina','ND':'North Dakota','OH':'Ohio','OK':'Oklahoma',
 'OR':'Oregon','PA':'Pennsylvania','RI':'Rhode Island','SC':'South Carolina',
 'SD':'South Dakota','TN':'Tennessee','TX':'Texas','UT':'Utah','VT':'Vermont',
 'VA':'Virginia','WA':'Washington','WV':'West Virginia','WI':'Wisconsin','WY':'Wyoming'}

# gross output of (region, industry) = column cost total = supply absorbed as cost
OUT = (Z.sum(0) + Mi + VA + taxes).reshape(n, S)   # (n, S) industry output proxy
print(f'loaded grav_fric_{VERSION} {YEAR}:  {n} regions x {S} sectors   '
      f'Z sum={Z.sum():.0f}  total output={OUT.sum():.0f} (10s of $bn)')


## Part A — finding characteristic `state × sector` economies

For every cell we compute a **location quotient** (LQ): how concentrated sector
*s* is in state *r* relative to the whole US.

$$\mathrm{LQ}_{r,s}=\frac{\text{out}_{r,s}/\sum_{s'}\text{out}_{r,s'}}
{\sum_{r'}\text{out}_{r',s}/\sum_{r',s'}\text{out}_{r',s'}}$$

LQ ≫ 1 means the state is far more specialised in that sector than the nation.
We keep only cells that are also *materially large* (output above the national
median for the sector) so we surface real specialisations, not rounding noise.
If the reconstruction is sound, the top of this list should read like an economic
atlas of the US.

In [ ]:
# ── Location quotient: surface the most characteristic state x sector cells ───
share_state = OUT / OUT.sum(1, keepdims=True)          # sector mix within a state
share_nat   = OUT.sum(0) / OUT.sum()                   # national sector mix
LQ = share_state / share_nat[None, :]                  # (n, S)

# size filter: sector output above its national median across states
thr = np.median(OUT, axis=0)[None, :]
big = OUT > thr

rows = []
for r in range(n):
    for s in range(S):
        if big[r, s]:
            rows.append((regions[r], sectors[s], DESC.get(sectors[s], sectors[s]),
                         LQ[r, s], OUT[r, s], 100 * share_state[r, s]))
char = (pd.DataFrame(rows, columns=['state', 'sector', 'description',
                                    'LQ', 'output', 'share_in_state_%'])
        .sort_values('LQ', ascending=False).reset_index(drop=True))

print('Top 25 most characteristic (specialised) state x sector cells:')
with pd.option_context('display.max_colwidth', 42, 'display.width', 160):
    print(char.head(25).to_string(index=False,
          formatters={'LQ': '{:.1f}'.format, 'output': '{:.1f}'.format,
                      'share_in_state_%': '{:.1f}'.format}))


In [ ]:
# ── LQ heatmap: states x a curated set of "telling" sectors ──────────────────
focus = ['oil','min','pet','che','mot','agr','fin','bnk','ins','sec','rnt','air','trn','trk',
         'com','dat','leg','alt']
focus = [s for s in focus if s in sidx]
sub   = LQ[:, [sidx[s] for s in focus]]

fig, ax = plt.subplots(figsize=(13, 11), dpi=130)
im = ax.imshow(np.clip(sub, 0, 4), cmap='RdBu_r', vmin=0, vmax=4, aspect='auto')
ax.set_xticks(range(len(focus))); ax.set_xticklabels([name(s) for s in focus],
                                                     rotation=90, fontsize=7)
ax.set_yticks(range(n)); ax.set_yticklabels(regions, fontsize=6)
ax.set_title(f'Location quotient (clipped at 4) — {YEAR}\n'
             'red = state over-specialised in that sector vs US average')
# annotate the strongly specialised cells
for i in range(n):
    for j in range(len(focus)):
        if sub[i, j] >= 2.5:
            ax.text(j, i, f'{sub[i, j]:.0f}', ha='center', va='center',
                    fontsize=5, color='k')
plt.colorbar(im, ax=ax, shrink=0.6, label='LQ')
plt.tight_layout(); plt.show()


### Per-sector view — who are the biggest producers?

Part A reads the table cell-by-cell (each `state × sector` ranked by LQ). Here we
flip the axis: for **every sector**, list the states with the **largest output**
in that good — the dominant producers — alongside their share of US output for
the sector and their location quotient. If the reconstruction is sound, each
row should name the states one expects (cars → MI, petroleum → LA/TX, casinos &
hotels → NV, farms → CA/IA, software → CA/WA, …).

In [ ]:
# ── For each sector: the states that produce the most of it ──────────────────
TOPK = 5                                       # number of leading states per sector

# tidy long table: rank-ordered top producers of every sector
records = []
for s in range(S):
    order = np.argsort(OUT[:, s])[::-1][:TOPK]
    for rank, ri in enumerate(order, 1):
        records.append((sectors[s], DESC.get(sectors[s], sectors[s]), rank,
                        regions[ri], OUT[ri, s],
                        100 * OUT[ri, s] / OUT[:, s].sum(), LQ[ri, s]))
top_states = pd.DataFrame(records, columns=[
    'sector', 'description', 'rank', 'state', 'output',
    'US_sector_share_%', 'LQ'])

# compact one-row-per-sector readout, biggest sectors (national output) first
sector_order = np.argsort(OUT.sum(0))[::-1]
lines = []
for s in sector_order:
    sub = top_states[top_states['sector'] == sectors[s]]
    leaders = '  '.join(f'{r}({o:.0f}, {p:.0f}%)'
                        for r, o, p in zip(sub['state'], sub['output'],
                                           sub['US_sector_share_%']))
    lines.append((sectors[s], DESC.get(sectors[s], sectors[s])[:34], leaders))
view = pd.DataFrame(lines, columns=[
    'sector', 'description', f'top {TOPK} producing states  (output 10s$bn, % of US)'])

print(f'Leading producer states by sector — output ranked, {YEAR}:')
with pd.option_context('display.max_colwidth', 90, 'display.width', 220,
                       'display.max_rows', None):
    print(view.to_string(index=False))


## Part B — full structure of a characteristic state

`plot_state(r)` draws nine panels for one state:

1. **intra-state Z** — the sector² block of within-state intermediate trade
   (row = good supplied, column = industry buying);
2. **exports** to every other state, by good (heatmap);
3. **imports** from every other state, by good (heatmap);
4. **production-cost decomposition** by industry (intermediates + imported
   intermediates + value added + taxes, which must stack up to gross output);
5. **final-demand decomposition** by good (C / I / G);
6. **tax decomposition** by type (output tax, tariffs, product tax on
   intermediates and on final use);
7. **specialisation** — the state's highest-LQ sectors;
8. **top trade partners** — gross exports vs imports by partner state;
9. a **summary** panel with the headline aggregates and balance residuals.

In [ ]:
# ── plot_state: twelve-panel structural portrait of one state ────────────────
def plot_state(r):
    ri = ridx[r]; x = np.arange(S)
    title = f'{STATE.get(r, r)} ({r}) — intra-US MRIOT structure, {YEAR}'
    fig = plt.figure(figsize=(22, 25), dpi=300)
    fig.suptitle(title, fontsize=17, y=0.995)

    # 1) intra-state sector^2 intermediate block
    ax = plt.subplot(4, 3, 1)
    im = ax.imshow(np.log1p(Z4[ri, :, ri, :]), cmap='viridis', aspect='auto')
    ax.set_title(f'{r}->{r} intra-state intermediate Z  (log1p)')
    ax.set_xlabel('buying industry'); ax.set_ylabel('selling good')
    ax.set_xticks(x); ax.set_xticklabels(sectors, rotation=90, fontsize=3.5)
    ax.set_yticks(x); ax.set_yticklabels(sectors, fontsize=3.5)
    plt.colorbar(im, ax=ax, shrink=0.7)

    # 2) exports to other states, by good  (partner state x good)
    others = [j for j in range(n) if j != ri]
    exp = (Z4[ri][:, others, :].sum(2) + F4[ri][:, others, :].sum(2)).T  # (n-1, S)
    ax = plt.subplot(4, 3, 2)
    im = ax.imshow(np.log1p(exp), cmap='magma', aspect='auto')
    ax.set_title(f'{r} interstate EXPORTS to other states, by good (log1p)')
    ax.set_xlabel('good'); ax.set_ylabel('destination state')
    ax.set_xticks(x); ax.set_xticklabels(sectors, rotation=90, fontsize=3.5)
    ax.set_yticks(range(len(others))); ax.set_yticklabels([regions[j] for j in others], fontsize=4)
    plt.colorbar(im, ax=ax, shrink=0.7)

    # 3) imports from other states, by good
    imp = np.array([Z4[j, :, ri, :].sum(1) + F4[j, :, ri, :].sum(1) for j in others])  # (n-1, S)
    ax = plt.subplot(4, 3, 3)
    im = ax.imshow(np.log1p(imp), cmap='cividis', aspect='auto')
    ax.set_title(f'{r} interstate IMPORTS from other states, by good (log1p)')
    ax.set_xlabel('good'); ax.set_ylabel('origin state')
    ax.set_xticks(x); ax.set_xticklabels(sectors, rotation=90, fontsize=3.5)
    ax.set_yticks(range(len(others))); ax.set_yticklabels([regions[j] for j in others], fontsize=4)
    plt.colorbar(im, ax=ax, shrink=0.7)

    # 4) foreign EXPORTS to rest of world, by good  (single ROW partner -> strip)
    ax = plt.subplot(4, 3, 4)
    fex = vec(EX, r)                                   # (S,)
    im = ax.imshow(np.log1p(fex)[:, None], cmap='magma', aspect='auto')
    ax.set_title(f'{r} foreign EXPORTS to ROW, by good (log1p)')
    ax.set_yticks(x); ax.set_yticklabels(sectors, fontsize=3.5)
    ax.set_xticks([0]); ax.set_xticklabels(['ROW'], fontsize=7)
    plt.colorbar(im, ax=ax, shrink=0.7)

    # 5) foreign IMPORTS from rest of world, by good  (single ROW partner -> strip)
    ax = plt.subplot(4, 3, 5)
    fim = vec(M, r)                                    # (S,)
    im = ax.imshow(np.log1p(fim)[:, None], cmap='cividis', aspect='auto')
    ax.set_title(f'{r} foreign IMPORTS from ROW, by good (log1p)')
    ax.set_yticks(x); ax.set_yticklabels(sectors, fontsize=3.5)
    ax.set_xticks([0]); ax.set_xticklabels(['ROW'], fontsize=7)
    plt.colorbar(im, ax=ax, shrink=0.7)

    # 6) production-cost decomposition by industry
    ax = plt.subplot(4, 3, 6)
    cZ = Z.sum(0).reshape(n, S)[ri]; mi = vec(Mi, r); va = vec(VA, r); tx = vec(taxes, r)
    ax.bar(x, cZ, color='#1f77b4', label='intermediates (Z)')
    ax.bar(x, mi, bottom=cZ, color='#9467bd', label='imported interm.')
    ax.bar(x, va, bottom=cZ + mi, color='#2ca02c', label='value added')
    ax.bar(x, tx, bottom=cZ + mi + va, color='#d62728', label='taxes')
    ax.set_title(f'{r}: production-cost composition by industry  (= gross output)')
    ax.set_xticks(x); ax.set_xticklabels(sectors, rotation=90, fontsize=3.5)
    ax.legend(fontsize=8)

    # 7) final demand by good (C / I / G)
    ax = plt.subplot(4, 3, 7)
    Fr = F4[:, :, ri, :].sum(0)   # (S, 3) — all-origin final demand absorbed by r
    ax.bar(x, Fr[:, 0], color='#1f77b4', label='C household')
    ax.bar(x, Fr[:, 1], bottom=Fr[:, 0], color='#ff7f0e', label='I investment')
    ax.bar(x, Fr[:, 2], bottom=Fr[:, 0] + Fr[:, 1], color='#2ca02c', label='G government')
    ax.set_title(f'{r}: final demand by good (C / I / G)')
    ax.set_xticks(x); ax.set_xticklabels(sectors, rotation=90, fontsize=3.5)
    ax.legend(fontsize=8)

    # 8) final demand by good, decomposed by origin (domestic / interstate / ROW)
    ax = plt.subplot(4, 3, 8)
    dom     = F4[ri, :, ri, :].sum(1)                      # (S,) own-state final demand
    us_all  = F4[:, :, ri, :].sum((0, 2))                  # (S,) all US-origin final demand
    inter   = us_all - dom                                 # from other states
    foreign = np.clip(vec(M, r) - vec(Mi, r), 0, None)     # ROW imports for final use
    ax.bar(x, dom, color='#1f77b4', label='domestic (own state)')
    ax.bar(x, inter, bottom=dom, color='#ff7f0e', label='from other states')
    ax.bar(x, foreign, bottom=dom + inter, color='#2ca02c', label='from ROW (foreign)')
    ax.set_title(f'{r}: final demand by good, by origin')
    ax.set_xticks(x); ax.set_xticklabels(sectors, rotation=90, fontsize=3.5)
    ax.legend(fontsize=8)

    # 9) tax decomposition by type
    ax = plt.subplot(4, 3, 9)
    blocks = [('output tax (ty0)', vec(tax_prod, r), '#1f77b4'),
              ('tariff (tm0)',     vec(tariff, r),   '#ff7f0e'),
              ('prod.tax interm (ta0)', vec(tls_int, r), '#2ca02c'),
              ('prod.tax final (ta0)',  vec(tls_fd, r),  '#d62728')]
    pos = np.zeros(S); neg = np.zeros(S)
    for lab, v, c in blocks:
        p = np.clip(v, 0, None); m = np.clip(v, None, 0)
        ax.bar(x, p, bottom=pos, color=c, label=lab); ax.bar(x, m, bottom=neg, color=c)
        pos += p; neg += m
    ax.axhline(0, color='k', lw=0.6)
    ax.set_title(f'{r}: tax decomposition by type (neg = subsidy)')
    ax.set_xticks(x); ax.set_xticklabels(sectors, rotation=90, fontsize=3.5)
    ax.legend(fontsize=8)

    # 10) specialisation: top-LQ sectors
    ax = plt.subplot(4, 3, 10)
    order = np.argsort(LQ[ri])[::-1][:15][::-1]
    ax.barh(range(15), LQ[ri][order], color='#8c564b')
    ax.set_yticks(range(15)); ax.set_yticklabels([name(sectors[s]) for s in order], fontsize=7)
    ax.axvline(1, color='k', lw=0.8, ls='--')
    ax.set_title(f'{r}: most specialised sectors (location quotient)'); ax.set_xlabel('LQ')

    # 11) top trade partners (gross, summed over goods)
    ax = plt.subplot(4, 3, 11)
    exp_by = exp.sum(1); imp_by = imp.sum(1)
    top = np.argsort(exp_by + imp_by)[::-1][:15]
    xs = np.arange(len(top)); w = 0.4
    ax.bar(xs - w/2, exp_by[top], w, color='#2ca02c', label='exports ->')
    ax.bar(xs + w/2, imp_by[top], w, color='#d62728', label='imports <-')
    ax.set_xticks(xs); ax.set_xticklabels([regions[others[j]] for j in top], rotation=90, fontsize=7)
    ax.set_title(f'{r}: top interstate trade partners'); ax.legend(fontsize=8)

    # 12) summary text
    ax = plt.subplot(4, 3, 12); ax.axis('off')
    out_r = OUT[ri]; gtop = sectors[int(out_r.argmax())]
    row_supply = (Z.sum(1) + F.sum(1) + EX).reshape(n, S)[ri]
    col_cost   = OUT[ri]
    txt = (f'{STATE.get(r, r)} ({r}) — aggregates [10s of $bn]\n\n'
           f'gross output      {out_r.sum():9.1f}\n'
           f'value added       {vec(VA, r).sum():9.1f}\n'
           f'final demand      {F4[:, :, ri, :].sum():9.1f}\n'
           f'foreign exports   {vec(EX, r).sum():9.1f}\n'
           f'foreign imports   {vec(M, r).sum():9.1f}\n'
           f'taxes             {vec(taxes, r).sum():9.1f}\n\n'
           f'interstate exports {exp.sum():8.1f}\n'
           f'interstate imports {imp.sum():8.1f}\n'
           f'net interstate     {exp.sum() - imp.sum():8.1f}\n\n'
           f'largest industry  {gtop} ({DESC.get(gtop, gtop)[:22]})\n'
           f'  output {out_r.max():.1f}  = {100*out_r.max()/out_r.sum():.1f}% of state\n\n')
    ax.text(0.0, 0.98, txt, fontsize=11, va='top', family='monospace')

    plt.tight_layout(rect=[0, 0, 1, 0.985]); plt.show()


### Characteristic states

A curated set whose economies are textbook-distinct — if the links are well
rebuilt, each portrait should match intuition (oil in TX/LA, cars in MI,
casinos/hotels in NV, farms in IA, aerospace & software in WA, finance in NY).

In [ ]:
CHARACTERISTIC = ['TX', 'LA', 'MI', 'NV', 'IA', 'WA', 'NY', 'CA','WY','GA','SD']
for st in CHARACTERISTIC:
    plot_state(st)


## Part C — numeric accounting checks

Plausible pictures must rest on consistent accounts. We verify the
reference-equilibrium identities at the state level and confirm the trade blocks
are internally coherent.

- **Row vs column.** Row total = supply of a *good* (`Z` row + `F` row +
  foreign exports); column total = cost of an *industry* (`Z` column + imported
  intermediates + VA + taxes). They differ only by the make-matrix byproduct
  asymmetry — checked nationally and reported per state.
- **Interstate trade closure.** Total exports of every state to all others must
  equal total imports of all others — the bilateral matrix has no leak.
- **Sign / domain checks.** `Z, F, VA, EX, M ≥ 0`; no NaNs.

In [ ]:
# ── Accounting checks ────────────────────────────────────────────────────────
print('=== domain checks ===')
for nm_, a in [('Z', Z), ('F', F), ('VA', VA), ('EX', EX), ('M', M)]:
    print(f'  {nm_:4s} min={a.min(): .4f}  any NaN={np.isnan(a).any()}')

print('\n=== row (good supply) vs column (industry cost) ===')
row_supply = (Z.sum(1) + F.sum(1) + EX)
col_cost   = (Z.sum(0) + Mi + VA + taxes)
print(f'  national row sum = {row_supply.sum():.1f}   col sum = {col_cost.sum():.1f}')
mask = col_cost > 1e-6
rel  = np.abs(row_supply - col_cost)[mask] / col_cost[mask]
print(f'  per-cell |row-col|/col : median {np.median(rel):.2%}  mean {rel.mean():.2%}'
      '  (residual = byproducts, see v3 markdown)')

print('\n=== interstate trade closure (off-diagonal Z + F) ===')
ZF = Z4.sum((1, 3)) + F4.sum((1, 3))         # (origin, dest) total interstate-able flow
off = ZF.copy(); np.fill_diagonal(off, 0)
exp_by_state = off.sum(1); imp_by_state = off.sum(0)
print(f'  total interstate exports = {exp_by_state.sum():.1f}'
      f'   imports = {imp_by_state.sum():.1f}'
      f'   (identical by construction: {np.isclose(exp_by_state.sum(), imp_by_state.sum())})')

print('\n=== per-state byproduct residual (mean |row-col|/col) ===')
rs = row_supply.reshape(n, S); cc = col_cost.reshape(n, S)
res = []
for i, r in enumerate(regions):
    mk = cc[i] > 1e-6
    res.append((r, np.mean(np.abs(rs[i] - cc[i])[mk] / cc[i][mk])))
res = pd.DataFrame(res, columns=['state', 'mean_rel_resid']).sort_values('mean_rel_resid')
print('  best 5 :', ', '.join(f'{r}={v:.1%}' for r, v in res.head(5).values))
print('  worst 5:', ', '.join(f'{r}={v:.1%}' for r, v in res.tail(5).values))


### Read-out

Treat each portrait as a falsifiable prediction: the largest industry, the
specialisation bars, and the import/export heatmaps should line up with what the
real state economy looks like. Where they do, the gravity + RAS reconstruction of
bilateral links is behaving; where a panel looks wrong (e.g. a land-locked farm
state showing heavy oil exports), it flags the cell to investigate in
[v3_construction.ipynb](v3_construction.ipynb).

## Focused study

### Nevada

In [ ]:
# ── How much of Nevada's interstate output goes to California? ───────────────
# Restricted to MATERIAL (goods-producing) sectors: services, trade margins,
# transport, utilities, construction, finance, real estate, government and other
# non-shippable / intangible items are excluded from the origin goods.

MATERIAL = {
    'agr', 'fof', 'oil', 'min',                          # primary
    'fbp', 'tex', 'alt', 'wpd', 'ppd', 'pri',            # light manufacturing
    'pet', 'che', 'pla', 'nmp',                          # chemical / mineral
    'pmt', 'fmt', 'mch', 'cep', 'eec',                   # metals / machinery / electronics
    'mot', 'ote', 'fpd', 'mmf',                          # vehicles / furniture / misc mfg
}
mat = np.array([sidx[s] for s in sectors if s in MATERIAL])   # material good indices

src = ridx['NV']
others = [ridx[r] for r in regions if r != 'NV']              # all states except NV
ca     = ridx['CA']

# Shipments of NV-produced material goods, by destination state
# (intermediate use Z + final demand F), summed over goods, industries, FD cats.
ship_interm = Z4[src][mat][:, :, :].sum(axis=(0, 2))         # (n,) over dest region
ship_final  = F4[src][mat][:, :, :].sum(axis=(0, 2))         # (n,) over dest region
ship_total  = ship_interm + ship_final                        # (n,) by dest state

out_to_others = ship_total[others].sum()                      # NV -> all other states
out_to_ca     = ship_total[ca]                                # NV -> California
pct           = 100 * out_to_ca / out_to_others

print(f"Nevada material output shipped to other states : {out_to_others:8.1f} (10s of $bn)")
print(f"  of which to California                       : {out_to_ca:8.1f}")
print(f"  => share going to California                 : {pct:6.2f}%")

# Context: California's rank and the leading destination states
order = np.argsort(ship_total)[::-1]
order = [i for i in order if i != src][:8]
print("\nTop destination states for NV material output (interstate):")
for i in order:
    print(f"  {regions[i]:>3}  {ship_total[i]:8.1f}  ({100*ship_total[i]/out_to_others:5.2f}%)")


In [ ]:
# ── How much of Nevada's interstate imports come from California? ────────────
# Same MATERIAL filter as above (services, trade, transport, utilities,
# construction, finance, real estate, government, ... excluded from the goods).

# Imports of material goods BY Nevada, by ORIGIN state
# (intermediate use Z + final demand F), summed over goods, industries, FD cats.
imp_interm = Z4[:, mat, src, :].sum(axis=(1, 2))             # (n,) by origin region
imp_final  = F4[:, mat, src, :].sum(axis=(1, 2))             # (n,) by origin region
imp_total  = imp_interm + imp_final                          # (n,) by origin state

imp_from_others = imp_total[others].sum()                    # all other states -> NV
imp_from_ca     = imp_total[ca]                              # California -> NV
pct_imp         = 100 * imp_from_ca / imp_from_others

print(f"Nevada material imports from other states    : {imp_from_others:8.1f} (10s of $bn)")
print(f"  of which from California                   : {imp_from_ca:8.1f}")
print(f"  => share coming from California             : {pct_imp:6.2f}%")

# Context: California's rank and the leading source states
order = np.argsort(imp_total)[::-1]
order = [i for i in order if i != src][:8]
print("\nTop source states for NV material imports (interstate):")
for i in order:
    print(f"  {regions[i]:>3}  {imp_total[i]:8.1f}  ({100*imp_total[i]/imp_from_others:5.2f}%)")
